This notebook covers the two ways to import a catalog to HATS format: 
- `lsdb.from_dataframe()` for smaller catalogs only (~1-2M rows) which can fit into memory as a single pandas DataFrame
- the `hats-import` pipeline for catalogs that don't fit into memory (more like ~1B rows); the data is read in from a list of files

**You will need:**

- to install the __[hats](https://github.com/astronomy-commons/hats)__, __[hats-import](https://github.com/astronomy-commons/hats-import)__, and __[lsdb](https://github.com/astronomy-commons/lsdb)__ packages and their dependencies 
- the files `select_cat_sample.py` and `hats_import_with_pipeline.py`
- time (minutes for a smaller catalog, hours for a larger catalog)
    - **Note: it is computationally expensive and time-consuming to fully import a catalog to HATS format, but this only needs to be done one time.  Once a catalog is HATS-ed, a user can quickly access it and apply filters/computations to it using the `lsdb` framework.*

**The catalog:**

This notebook is meant to run on Aurora and uses the ATLAS-Refcat2 stellar reference catalog, select columns of which are stored on Aurora in a non-HATS-ed format in the following directory structure:
```
level-0 |- /etc/rico/atlas_refcat2/
level-1    |- 0/ 
level-2    |  |- 0/
           |  |  |- part-0.feather (same file name name in each level 2 directory; contains sources from 4 HEALPix of NSIDE=32)
level-2    |  |- 1/
           |  |...
level-2    |  |- 15/
level-1    |- 1/
           |...
level-1    |-191/ (given 12,288 healpix total of NSIDE=32, 4 pix per .feather file, 1 file per level-2 directory, 16 level-2s per level-1 directory gives 192 level-1 directories)
```

**Selecting a small "toy" catalog to try out these importing methods:**

This notebook reads in ATLAS data from those `.feather` files, and will select a small sample of those sources to demonstrate how an import can be performed.  This is done in two ways:
- If using `lsdb.from_dataframe()`, you can use function `get_rand_cat_sample()` (found in `select_cat_sample.py`) to select a fraction of ATLAS sources from all `.feather` files (evenly distributed across sky) and serve them up in a pandas DataFrame 
- Alternately, if reading data in from files when using the `hats-import` pipeline, you can simply elect to read from a certain number of the `.feather` files (out of the 3072 total) if you're not trying to import the entire ~1B-row catalog 
    - There is a class `FeatherReader()` (found in `hats_import_with_pipeline.py`) that can be provided to the `hats-import` pipeline, allowing it to read in data from `.feather` files

In [ ]:
# Imports
from dask.distributed import Client
import glob
from hats_import.catalog.arguments import ImportArguments
from hats_import.pipeline import pipeline,pipeline_with_client
import healpy as hp
import numpy as np
import os
import pandas as pd
import pyarrow as pa
import sys
import time

# custom functions, etc. by Katie 
from select_cat_sample import get_rand_cat_sample
from hats_import_with_pipeline import FeatherReader, import_pipeline , import_pipeline_test

# lsdb.from_dataframe()

# hats-import pipeline

The `hats_import_with_pipeline.py` script contains the same code/functionality.

In [ ]:
def import_pipeline_with_client(client,lst,cat_outpath,cat_outname,filereader,lo_hporder=2,hi_hporder=10,pix_thresh=5000):
    '''
    Import a catalog to HATS format with hats-import.pipeline_with_client() and a given Dask Client
    
    Arguments
    ---------
        lst: (str list) list of files with cat data in non-hatsed format; expects "ra" and "dec" columns
        cat_outpath: (str) Full path to directory where HATSed cat will be written
        cat_outname: (str) Name of HATSed cat
        filereader: (FileReader Object) File Reader to pull in un-HATS-ed cat data (i.e. FeatherReader)
        lo_hporder: (int, Default = 2) Lowest-order HPix for tiling
        hi_hporder: (int, Default = 10) Highest-order HPix for tiling
        pix_thresh: (int, Default = 5000) Max #sources per tile

    Returns
    --------
        Nothing, HATSed cat is written to cat_outpath/cat_outname
    '''
    args = ImportArguments(
        ra_column="ra",
        dec_column="dec",
        input_file_list=lst,
        file_reader=filereader,
        output_artifact_name=cat_outname,
        output_path=cat_outpath,
        lowest_healpix_order=lo_hporder,
        highest_healpix_order=hi_hporder,
        pixel_threshold=pix_thresh,
        )
    pipeline_with_client(args,client)

In [9]:
# Setup

# catalog name
cat = "atlas" 
cat_name = "ATLAS-Refcat2"

# directory on disk containing un-HATS-ed catalog
cat_inpath = "/etc/rico/atlas_refcat2/"

# Files containing un-HATS-ed catalog, on disk
cat_infiles = "*/*/*.feather" 

# Number of files to read in 
nfiles = 500 # if you want to read all files (unadvisable!), set to 0

# catalog output path (where  HATS-ed cat will go)
cat_outpath = "/net/scratch/kmfas/atlas_refcat2/" 

# catalog output name (name of HATASed cat)
cat_outname = "atlas_"+str(nfiles)+"files" #args.cat_outname[0]

# Inport arguments (to feed to hats-import pipeline)
lo_hp = 2 # lowest-order HEALPix
hi_hp = 12 # highets-order HEALPix
pix_thrsh = 5000 # max number of sources per HATS pixel/partition 
n_dask_workers = 1 # number of dask workers
n_dask_threads = 1 # number of threads per dask worker

# Get a list of files containing the un-HATSed catalog
lst = glob.glob(cat_inpath+cat_infiles,recursive=True)
ntot = len(lst)
if nfiles!=0: lst = lst[:nfiles]

Try importing with `hats-import.pipeline()`, which creates a default Dask Client for which you can specify the number of workers and threads (don't actually do this on Aurora, as Dask will allocate total system RAM to its workers):

In [ ]:
# Import the catalog with pipeline
print(f"Importing catalog {cat_name} to HATS format....")
if nfiles!=0: print(f"...only from {nfiles}/{ntot} files in {cat_inpath}")
else: print(f"...from all {ntot} files in {cat_inpath}")
print(f"HATSed catalog will be written to {cat_outpath+cat_outname}")
print("---------------------------------------")
import_pipeline(lst,cat_outpath,cat_outname,FeatherReader(),lo_hp,hi_hp,pix_thrsh,n_dask_workers,n_dask_threads)
print("---------------------------------------")
print("Catalog imported?")

Importing catalog ATLAS-Refcat2 to HATS format....
...only from 100/3072 files in /etc/rico/atlas_refcat2/
HATSed catalog will be written to /net/scratch/kmfas/atlas_refcat2/atlas_100files
---------------------------------------


Catalog: Planning  :   0%|          | 0/4 [00:00<?, ?it/s]

Catalog: Mapping   :   0%|          | 0/100 [00:00<?, ?it/s]

Catalog: Binning   :   0%|          | 0/2 [00:00<?, ?it/s]

Catalog: Splitting :   0%|          | 0/100 [00:00<?, ?it/s]

Catalog: Reducing  :   0%|          | 0/4936 [00:00<?, ?it/s]

Catalog: Finishing :   0%|          | 0/6 [00:00<?, ?it/s]

---------------------------------------
Catalog imported?


Now try importing with `hats-import.pipeline_with_client`, which allows you to create and provide a custom Dask Client for which you can define the number of workers and threads *and* the memory limit per worker (the Right Way):

In [ ]:
# Import the catalog with pipeline_with_client
print(f"Importing catalog {cat_name} to HATS format....")
if nfiles!=0: print(f"...only from {nfiles}/{ntot} files in {cat_inpath}")
else: print(f"...from all {ntot} files in {cat_inpath}")
print(f"HATSed catalog will be written to {cat_outpath+cat_outname}")
print("---------------------------------------")

# if you don't want a persistent Client:
#with Client(n_workers=4,memory_limit="10GiB",threads_per_worker=1) as client:
#        imprt_pipeline_with_client(client,lst,cat_outpath,cat_outname,FeatherReader(),lo_hp,hi_hp,pix_thrsh)

# if you do want a persistent Client:
client = Client(n_workers=6,memory_limit="10GiB",threads_per_worker=1)
imprt_pipeline_with_client(client,lst,cat_outpath,cat_outname,FeatherReader(),lo_hp,hi_hp,pix_thrsh)
#client.close() remember to close the client before defining another one

print("---------------------------------------")
print("Catalog imported?")


Importing catalog ATLAS-Refcat2 to HATS format....
...only from 500/3072 files in /etc/rico/atlas_refcat2/
HATSed catalog will be written to /net/scratch/kmfas/atlas_refcat2/atlas_500files
---------------------------------------


/home/kmfas/argvenv/lib/python3.12/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 35985 instead
  warnings.warn(


Catalog: Planning  :   0%|          | 0/4 [00:00<?, ?it/s]

tmp_path (/net/scratch/kmfas/atlas_refcat2/atlas_500files/intermediate) contains intermediate files; resuming prior progress.


Catalog: Binning   :   0%|          | 0/2 [00:00<?, ?it/s]

Catalog: Splitting :   0%|          | 0/90 [00:00<?, ?it/s]

Catalog: Reducing  :   0%|          | 0/74768 [00:02<?, ?it/s]

Catalog: Finishing :   0%|          | 0/6 [00:00<?, ?it/s]

---------------------------------------
Catalog imported?


In [40]:
client.get_scheduler_logs()

(('INFO',
  '2026-05-15 17:23:18,504 - distributed.scheduler - INFO - Lost all workers'),
 ('WARNING',
  "2026-05-15 17:23:18,488 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:37361' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {'split_pixels-cfcee648177f95a20ac02e1c64cd5514', 'split_pixels-faa1ba93840070a73eac1117998ee78b', 'split_pixels-79cfb7ca99c458e16437fc69af649326', 'split_pixels-0c6ee748b707246f38740a6dd6cbbc9a', 'split_pixels-f75993bd6b8359a80a52b0bf6b54a185', 'split_pixels-f4bff79d1ba129d80f7c0aaf84cd4433', 'split_pixels-059096a6f32b49b6d586254e037dc5ff', 'split_pixels-18cf261413d1d298e59fad886dd360a0', 'split_pixels-dbe1a6878735b7d4c6c6314da1c7f9b4', 'split_pixels-bbc974e66eaab4f78cecf28cfa933de1', 'split_pixels-133c5e7b822c13a64fae343f3d9a9a4d', 'split_pixels-b07c7c04f40b3e7d330532e3f8f49ee1', 'split_pixels-aeedb1185c066b76b72fb2aafac9c357', 'split_pixels-2e9629ef19ede93e61df28d3f92d7a7b', 'split_pixels-9c1b

2026-05-15 17:23:31,541 - distributed.nanny - ERROR - Worker process died unexpectedly
Process Dask Worker process (from Nanny):
Traceback (most recent call last):
  File "/home/kmfas/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/asyncio/runners.py", line 118, in run
    return self._loop.run_until_complete(task)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/kmfas/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/asyncio/base_events.py", line 691, in run_until_complete
    return future.result()
           ^^^^^^^^^^^^^^^
  File "/home/kmfas/argvenv/lib/python3.12/site-packages/distributed/nanny.py", line 985, in run
    await worker.finished()
  File "/home/kmfas/argvenv/lib/python3.12/site-packages/distributed/core.py", line 494, in finished
    await self._event_finished.wait()
  File "/home/kmfas/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/asyncio/locks.py", line 212, in wait
    await fut
async

In [44]:
client.close()